In [ ]:
import akshare as ak
import pandas as pd
import warnings
stock_spot = ak.stock_zh_a_spot_em()  # 东方财富最新接口

In [ ]:

stock_spot = ak.stock_zh_a_spot_em()  # 东方财富最新接口


In [ ]:
#使用akshare 筛选股票

import akshare as ak
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ==========================
# 你的筛选规则
# ==========================
CONDITIONS = {
    "PE分位_min": 10,
    "PE分位_max": 20,
    "PB分位_min": 10,
    "PB分位_max": 20,
    "连续分红年数": 5,
    "股息率>": 2.0,
    "市值>": 50,
    "负债率<": 70,
}

# ==========================
# 1. 获取全A股数据（最新接口）
# ==========================
def get_data():
    print("正在获取全A股数据...")

    # 股票列表 + 行业
    stock_spot = ak.stock_zh_a_spot_em()  # 东方财富最新接口
    stock_spot = stock_spot[["代码", "名称", "行业", "市盈率-动态", "市净率", "总市值", "股息率"]]

    # 估值分位数据
    try:
        pe_pb = ak.stock_a_pe_and_pb_lg()  # 最新可用估值分位
    except:
        pe_pb = ak.stock_a_pe_pb()

    # 财务数据
    fina = ak.stock_financial_report()

    # 合并
    df = stock_spot.merge(pe_pb[["代码", "pe_percent", "pb_percent"]], on="代码", how="left")
    df = df.merge(fina[["代码", "资产负债率", "净利润同比"]], on="代码", how="left")

    # 分红数据
    div = ak.stock_dividend_yield()
    df = df.merge(div, on="代码", how="left")

    return df

# ==========================
# 2. 核心筛选（你的策略）
# ==========================
def filter_strategy(df):
    print("开始筛选：低估值 + 分红稳 + 行业好 + 排雷")

    # 市值过滤
    df = df[df["总市值"] >= CONDITIONS["市值>"]]

    # 估值历史分位 10%~20%
    df = df[(df["pe_percent"] >= CONDITIONS["PE分位_min"]) &
            (df["pe_percent"] <= CONDITIONS["PE分位_max"]) &
            (df["pb_percent"] >= CONDITIONS["PB分位_min"]) &
            (df["pb_percent"] <= CONDITIONS["PB分位_max"])]

    # 分红稳定
    df = df[df["股息率"] >= CONDITIONS["股息率>"]]

    # 排雷
    df = df[df["资产负债率"] <= CONDITIONS["负债率<"]]
    df = df[df["净利润同比"] > 0]

    # 好行业
    good_ind = [
        "银行", "电力", "煤炭", "建筑装饰", "公用事业",
        "家电", "证券", "保险", "铁路", "公路", "水务"
    ]
    df = df[df["行业"].isin(good_ind)]

    # 剔除ST
    df = df[~df["名称"].str.contains("ST|退|*", na=False)]

    # 排序
    df = df.sort_values("股息率", ascending=False)
    return df

# ==========================
# 3. 主程序
# ==========================
def main():
    df = get_data()
    res = filter_strategy(df)

    print("\n" + "="*70)
    print(f"✅ 筛选完成：共 {len(res)} 只符合条件股票")
    print("="*70)

    cols = ["代码","名称","行业","市盈率-动态","市净率","pe_percent","pb_percent","股息率","总市值"]
    print(res[cols].to_string(index=False))

    # 保存Excel
    res.to_excel("低估值高股息筛选结果.xlsx", index=False)
    print("\n📄 已保存到：低估值高股息筛选结果.xlsx")

if __name__ == "__main__":
    main()

In [ ]:
import akshare as ak

# 获取 A 股所有股票的最新实时行情
stock_zh_a_spot_df = ak.stock_zh_a_spot_em()
print(stock_zh_a_spot_df)

# 获取单只股票的历史日K线（以历史数据为例）
stock_hist_df = ak.stock_zh_a_hist(symbol="600519", period="daily", start_date="20250101", end_date="20260515", adjust="qfq")
# 保存为 Excel
stock_hist_df.to_excel("600519_history.xlsx", index=False)

In [4]:
import akshare as ak

# 新版新浪 A 股实时行情接口
stock_zh_a_spot_df = ak.stock_zh_a_market_item()
print(stock_zh_a_spot_df)

AttributeError: module 'akshare' has no attribute 'stock_zh_a_market_item'